# **Annotation and Model Evaluation**

In this session, we will explore the process of evaluating annotation results, including inter-annotator agreement and evaluation of model performance using metrics such as accuracy, precision, recall, and F1-score.

## Annotation Agreement

When multiple annotators are involved in labeling data, it is crucial to assess the consistency of their annotations. This is typically done using inter-annotator agreement metrics, such as Cohen's Kappa or Fleiss' Kappa, which measure the level of agreement between annotators beyond what would be expected by chance.

Cohen's Kappa is used when there are two annotators, while Fleiss' Kappa is used for more than two annotators. These metrics provide a value between -1 and 1, where 1 indicates perfect agreement, 0 indicates agreement equivalent to chance, and negative values indicate less agreement than expected by chance.

If we have 3 annotators, we can calculate the Cohen's Kappa for each pair of annotators, and then average the Kappa values to get an overall agreement score. Alternatively, we can use Fleiss' Kappa to directly calculate the agreement among all annotators.

In [6]:
import pandas as pd

annotated_path = "../dataset/annotation.csv"
annotated_df = pd.read_csv(annotated_path)
annotated_df.head()

,ann1,ann2,ann3
0,neutral,neutral,negative
1,positive,negative,positive
2,positive,positive,neutral
3,neutral,negative,neutral
4,negative,negative,negative


In [7]:
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters


def pairwise_kappa(annotators: list[list[str]]) -> float:
    kappa_scores = []
    for i in range(len(annotators)):
        for j in range(i + 1, len(annotators)):
            kappa = cohen_kappa_score(annotators[i], annotators[j])
            kappa_scores.append(kappa)
    return sum(kappa_scores) / len(kappa_scores)


def fleiss_kappa_score(annotators: list[list[str]]) -> float:
    # transpose the list of annotators to get the format required by fleiss_kappa
    annotators_transposed = list(zip(*annotators))

    table, _ = aggregate_raters(annotators_transposed)
    return fleiss_kappa(table)

With above functions we created the pairwise Cohen's Kappa and Fleiss' Kappa calculations, now we can apply these functions to our annotation data to evaluate the agreement among annotators.

In [8]:
ann1 = annotated_df["ann1"].tolist()
ann2 = annotated_df["ann2"].tolist()
ann3 = annotated_df["ann3"].tolist()

cohen_score = pairwise_kappa([ann1, ann2, ann3])
fleiss_score = fleiss_kappa_score([ann1, ann2, ann3])

print(f"Pairwise Cohen's Kappa: {cohen_score:.4f}")
print(f"Fleiss' Kappa: {fleiss_score:.4f}")

Pairwise Cohen's Kappa: 0.0013
Fleiss' Kappa: 0.0012


The score for Cohen's 0.0013 mean that there is very low agreement between the annotators, while the Fleiss' Kappa score of 0.0012 also indicates a very low level of agreement among the annotators. This suggests that the annotations may be inconsistent and may require further review or clarification of the annotation guidelines.

## Annotators Aggregation

As we have three annotators, we can also consider aggregating their annotations to create a single "consensus" annotation for each data point. This can be done using majority voting, where the most common annotation among the annotators is selected as the final annotation for each data point. This approach can help to mitigate the effects of individual annotator bias and improve the overall quality of the annotations.

In [9]:
# function to get majority vote
def majority(annotations: list[str]) -> str:
    return max(set(annotations), key=annotations.count)


annotated_df["final_label"] = annotated_df[["ann1", "ann2", "ann3"]].apply(
    lambda row: majority(row.tolist()), axis=1
)
annotated_df.head()

,ann1,ann2,ann3,final_label
0,neutral,neutral,negative,neutral
1,positive,negative,positive,positive
2,positive,positive,neutral,positive
3,neutral,negative,neutral,neutral
4,negative,negative,negative,negative


In [10]:
# save the annotated dataframe with final labels
aggregated_path = "../dataset/annotation_with_final_label.csv"
annotated_df.to_csv(aggregated_path, index=False)

## Model Evaluation

Now we have our final labels from the annotators, we can proceed to evaluate the performance of our model using these labels as the ground truth. We can use various evaluation metrics such as accuracy, precision, recall, and F1-score to assess how well our model is performing in classifying the data points according to the final labels.

In [19]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


def evaluate_perlabel(
    y_true: list[str],
    y_pred: list[str],
    label: str,
) -> dict[str, float]:

    y_true_binary = [1 if y == label else 0 for y in y_true]
    y_pred_binary = [1 if y == label else 0 for y in y_pred]

    accuracy = accuracy_score(y_true=y_true_binary, y_pred=y_pred_binary)
    precision = precision_score(y_true=y_true_binary, y_pred=y_pred_binary)
    recall = recall_score(y_true=y_true_binary, y_pred=y_pred_binary)
    f1 = f1_score(y_true=y_true_binary, y_pred=y_pred_binary, pos_label=1)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
    }


def evaluate_multiclass(
    y_true: list[str],
    y_pred: list[str],
) -> dict[str, float]:

    accuracy = accuracy_score(y_true=y_true, y_pred=y_pred)
    f1_weighted = f1_score(y_true=y_true, y_pred=y_pred, average="weighted")
    f1_macro = f1_score(y_true=y_true, y_pred=y_pred, average="macro")

    return {
        "accuracy": accuracy,
        "weighted_f1": f1_weighted,
        "macro_f1": f1_macro,
    }

In [17]:
inference_path = "../dataset/inference_result.csv"
inference_df = pd.read_csv(inference_path)
inference_df.head()

,label
0,negative
1,negative
2,neutral
3,negative
4,positive


In [23]:
# Calculate evaluation metrics
y_true = annotated_df["final_label"].tolist()
y_pred = inference_df["label"].tolist()

# Evaluate per label
print("Per Label Evaluation:")
labels = set(y_true)
for label in labels:
    print(f"Label: {label}")
    metrics = evaluate_perlabel(y_true, y_pred, label)
    for metric_name, metric_value in metrics.items():
        print(f"{metric_name.capitalize()}: {metric_value:.4f}")
    print()

# Evaluate overall
print("Overall Evaluation:")
overall_metrics = evaluate_multiclass(y_true, y_pred)
for metric_name, metric_value in overall_metrics.items():
    print(f"{metric_name.capitalize()}: {metric_value:.4f}")

Per Label Evaluation:
Label: neutral
Accuracy: 0.5100
Precision: 0.5037
Recall: 0.3694
F1_score: 0.4262

Label: positive
Accuracy: 0.5933
Precision: 0.2562
Recall: 0.3271
F1_score: 0.2874

Label: negative
Accuracy: 0.5673
Precision: 0.2238
Recall: 0.2779
F1_score: 0.2480

Overall Evaluation:
Accuracy: 0.3353
Weighted_f1: 0.3457
Macro_f1: 0.3205
